# Is there enough data for pooled and single-family SPB residual models?

**Purpose:** audit experimental Seebeck/carrier data before fitting symbolic regression.
This notebook compares a **pooled model across families** with **one selected family** (default: PbTe).
It does not substitute a prescribed DFT concentration for an experimental concentration, fit a mass
from the target Seebeck value, or claim that an arbitrary sample count guarantees successful SR.

## Run in Google Colab
1. Upload this notebook to Colab. A CPU runtime is sufficient.
2. In the configuration cell, select your family and data source.
3. Run all cells. With the default **upload** mode, upload these two existing files:
   - `raw_starrydata2_samples.csv.gz` (or `ThermoelectricMaterials_samples.csv.gz`)
   - `raw_starrydata2_curves.csv.gz` (or `ThermoelectricMaterials_curves.csv.gz`)
   - Optionally, `raw_starrydata2_manifest.json` to verify snapshot hashes.
4. Read the candidate-sample funnel, family comparison, study concentration, and mass coverage.
5. Download the results ZIP from the final cell. Send back `family_comparison.csv`,
   `scope_summary.csv`, `audit_report.md`, and `run_manifest.json` for interpretation.

**Count meanings:** a curve point is not an independent sample; a sample ID is not proof of a
unique experimental specimen. This audit counts distinct sample IDs and source DOIs separately.
Final source-paper verification must resolve duplicate specimens, theoretical curves, specimen identity,
Hall-factor interpretation, phase, orientation, and mass provenance.

In [ ]:
# 1. Configuration: change these values before Run all.
DATA_SOURCE = "upload"                 # "upload" or "download"
SELECTED_FAMILY = "PbTe"                # e.g. "Half-Heusler", "Bi2Te3", "Skutterudite"
POOLED_FAMILIES = None                   # None = all available tags; or ["PbTe", "Bi2Te3", "Half-Heusler"]
INCLUDE_UNTAGGED_IN_POOL = False         # avoids calling an unidentified mixture a general TE family pool
REQUIRE_EXPLICIT_EXPERIMENT_TAG = False # False retains unknown tags but reports them separately

TEMPERATURE_TOLERANCE_K = 5.0            # nearest measured points; report sensitivity below
TEMPERATURE_MIN_K = 200.0                # explicit study domain, not a universal quality rule
TEMPERATURE_MAX_K = 1200.0
ROOM_TEMPERATURE_K = 300.0
ROOM_WINDOW_K = 10.0
MAX_RELATIVE_CARRIER_DISAGREEMENT = 0.25 # flags competing curves near the same temperature

# Broad screening limits, not literature-derived proof that values outside them are wrong.
MIN_CARRIER_CM3 = 1e14
MAX_CARRIER_CM3 = 1e23
MAX_ABS_SEEBECK_UVK = 2000.0

# Optional independent mass table. Upload only if you have genuinely independent DOS masses.
UPLOAD_INDEPENDENT_MASS_TABLE = False

# Optional automatic source. Download mode resolves a release once, keeping all assets in one snapshot.
GITHUB_REPOSITORY = "atsumitanaka/starrydata-datasets"
RELEASE_TAG = "latest"                   # pin a known release tag for exact reproducibility

In [ ]:
# 2. Libraries and directories. Standard Google Colab packages; no paid APIs or GPU.
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import ast, hashlib, json, re, zipfile
import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

ROOT = Path('/content/spb_data_audit') if Path('/content').exists() else Path('spb_data_audit')
RAW = ROOT / 'raw'; OUT = ROOT / 'results'
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 80)
assert DATA_SOURCE in {'upload', 'download'}
assert 0 <= TEMPERATURE_TOLERANCE_K <= 25
assert TEMPERATURE_MIN_K < TEMPERATURE_MAX_K
run_info = {'run_utc':datetime.now(timezone.utc).isoformat(), 'source_mode':DATA_SOURCE,
            'scope':'availability audit, not source-verified training data'}

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def normalize_doi(v):
    if pd.isna(v): return ''
    return re.sub(r'^(https?://(dx\.)?doi\.org/|doi:\s*)', '', str(v).strip().lower())

def norm(v):
    return re.sub(r'\s+', ' ', str(v).strip().lower()) if pd.notna(v) else ''

def read_json_dict(v):
    if not isinstance(v,str): return {}
    try:
        d=json.loads(v); return d if isinstance(d,dict) else {}
    except (ValueError,TypeError): return {}

def category(info, key):
    value=info.get(key,{})
    if isinstance(value,dict): return str(value.get('category') or '').strip()
    return str(value or '').strip()

def markdown_table(frame):
    def escaped(v): return str(v).replace('|', r'\|').replace('\n', ' ')
    rows=[list(frame.columns), ['---']*len(frame.columns)] + frame.astype(str).values.tolist()
    return '\n'.join('| '+' | '.join(map(escaped,row))+' |' for row in rows)

In [ ]:
# 3. Load a reproducible raw snapshot. The upload option accepts .csv as well as .csv.gz.
if DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded=files.upload()
    paths=[]
    for name, data in uploaded.items():
        path=RAW / Path(name).name
        path.write_bytes(data); paths.append(path)
    def select_upload(word):
        hits=[p for p in paths if word in p.name.lower() and
              (p.name.endswith('.csv.gz') or p.name.endswith('.csv'))]
        if len(hits)!=1:
            raise ValueError(f'Upload exactly one RAW {word} CSV/CSV.GZ; found {[p.name for p in hits]}. '
                             'Use raw_starrydata2_samples and raw_starrydata2_curves, not family summaries.')
        return hits[0]
    sample_path=select_upload('samples'); curve_path=select_upload('curves')
    manifests=[p for p in paths if 'manifest' in p.name.lower() and p.suffix=='.json']
    source_manifest=json.loads(manifests[0].read_text()) if manifests else None
else:
    api=f'https://api.github.com/repos/{GITHUB_REPOSITORY}/releases/'
    api += 'latest' if RELEASE_TAG=='latest' else 'tags/'+RELEASE_TAG
    response=requests.get(api,timeout=60)
    if not response.ok:
        raise RuntimeError(f'Release lookup failed ({response.status_code}). Switch DATA_SOURCE to upload '
                           'and use your two existing raw files; do not substitute another dataset silently.')
    release=response.json()
    assets={a['name']:a for a in release.get('assets',[])}
    run_info['release']={'repository':GITHUB_REPOSITORY,'tag':release['tag_name'],'url':release['html_url']}
    def download_asset(name):
        if name not in assets: raise ValueError(f'{name} missing in this release. Available: {list(assets)}')
        dest=RAW/name
        with requests.get(assets[name]['browser_download_url'],stream=True,timeout=(30,180)) as r:
            r.raise_for_status()
            with open(dest,'wb') as f:
                for chunk in r.iter_content(1024*1024): f.write(chunk)
        print('Downloaded',name, f'{dest.stat().st_size/1e6:.1f} MB')
        return dest
    sample_path=download_asset('ThermoelectricMaterials_samples.csv.gz')
    curve_path=download_asset('ThermoelectricMaterials_curves.csv.gz')
    source_manifest=None
    if 'manifest.json' in assets:
        source_manifest=json.loads(download_asset('manifest.json').read_text())

samples=pd.read_csv(sample_path,low_memory=False)
curves=pd.read_csv(curve_path,low_memory=False)
required_s={'sample_id','DOI','composition','sample_info'}
required_c={'sample_id','DOI','prop_x','prop_y','unit_x','unit_y','x','y'}
assert required_s.issubset(samples.columns), f'Missing sample fields: {required_s-set(samples.columns)}'
assert required_c.issubset(curves.columns), f'Missing curve fields: {required_c-set(curves.columns)}'
run_info['inputs']={p.name:{'bytes':p.stat().st_size,'sha256':sha256(p)} for p in [sample_path,curve_path]}
if source_manifest:
    run_info['snapshot']=source_manifest.get('db_snapshot',source_manifest.get('generated_at'))
    expected=source_manifest.get('projects',{}).get('ThermoelectricMaterials',{})
    for kind,path in [('samples',sample_path),('curves',curve_path)]:
        check=expected.get(kind,{}).get('sha256')
        if check and check!=sha256(path):
            raise ValueError(f'{kind} hash does not match the supplied manifest: do not mix snapshots.')
print(f'{len(samples):,} sample records; {len(curves):,} curve records')
print('Snapshot:',run_info.get('snapshot','not supplied; file hashes recorded'))

## Metadata and eligibility
Families are taken from the database's explicit `MaterialFamily` tag. A tag is a candidate label,
not a verified crystal structure. **No substring matching of chemical formulas is used:** a material
containing Sn and Se is not automatically SnSe.

Explicit theory/calculation records are excluded. Untagged experimental status is retained by default
and clearly counted; change `REQUIRE_EXPLICIT_EXPERIMENT_TAG` to require positive experimental labels.
Empty or malformed source identifiers are excluded from matched-pair counts.

In [ ]:
# 4. Normalize metadata without overwriting source files.
samples=samples.copy(); curves=curves.copy()
for frame in [samples,curves]:
    frame['sample_id']=frame['sample_id'].astype(str).str.replace(r'\.0$','',regex=True)
    frame['DOI']=frame['DOI'].map(normalize_doi)
if samples.sample_id.duplicated().any():
    samples[samples.sample_id.duplicated(False)].to_csv(OUT/'duplicate_sample_ids.csv',index=False)
    raise ValueError('Sample IDs are not unique; inspect duplicate_sample_ids.csv before joining.')
infos=samples.sample_info.map(read_json_dict)
samples['family']=infos.map(lambda d:category(d,'MaterialFamily')).replace('', 'Untagged')
samples['process']=infos.map(lambda d:category(d,'FabricationProcess')).replace('', 'Unknown')
samples['data_type']=infos.map(lambda d:category(d,'DataType')).replace('', 'Unknown')
samples['form']=infos.map(lambda d:category(d,'Form')).replace('', 'Unknown')
samples['purity']=infos.map(lambda d:category(d,'Purity')).replace('', 'Unknown')
types=samples.data_type.str.lower()
samples['explicit_experiment']=types.str.contains('experiment',na=False)
samples['explicit_theory']=types.str.contains('theor|calcul|simulat|dft',regex=True,na=False)
samples['eligible_metadata']=~samples.explicit_theory
if REQUIRE_EXPLICIT_EXPERIMENT_TAG:
    samples['eligible_metadata'] &= samples.explicit_experiment
samples['doi_valid']=samples.DOI.str.match(r'^10\.\d{4,9}/\S+$',na=False)
samples['eligible_metadata'] &= samples.doi_valid
samples['pool_included']=samples.eligible_metadata
if POOLED_FAMILIES is not None:
    samples['pool_included'] &= samples.family.isin(POOLED_FAMILIES)
if not INCLUDE_UNTAGGED_IN_POOL:
    samples['pool_included'] &= samples.family.ne('Untagged')
samples['selected_family']=samples.family.str.casefold().eq(SELECTED_FAMILY.casefold())
print('Experimental-status labels:'); display(samples.data_type.value_counts().head(20).to_frame('samples'))
display(samples.groupby('family').agg(samples=('sample_id','nunique'),papers=('DOI','nunique'))
        .sort_values('samples',ascending=False).head(30))
if not samples.selected_family.any():
    raise ValueError(f'Family {SELECTED_FAMILY!r} was not found. Select an exact tag from the table.')

In [ ]:
# 5. Broad availability audit: presence alone is NOT a matched measurement.
curves['curve_id']=np.arange(len(curves))
curves['px']=curves.prop_x.map(norm); curves['py']=curves.prop_y.map(norm)
property_audit=curves.groupby(['prop_x','unit_x','prop_y','unit_y'],dropna=False).agg(
    curves=('curve_id','size'),samples=('sample_id','nunique')).reset_index()
property_audit.to_csv(OUT/'property_axis_unit_audit.csv',index=False)
s_ids=set(curves.loc[curves.py.eq('seebeck coefficient'),'sample_id'])
n_names={'carrier concentration','hall density','carrier density n','hall concentration'}
n_ids=set(curves.loc[curves.py.isin(n_names|{'hall coefficient'}) | curves.px.isin(n_names),'sample_id'])
samples['has_S_any_axis']=samples.sample_id.isin(s_ids)
samples['has_carrier_or_Hall_any_axis']=samples.sample_id.isin(n_ids)
samples['broad_overlap']=samples.has_S_any_axis & samples.has_carrier_or_Hall_any_axis
mass_mask=(curves.px+' '+curves.py).str.contains('effective mass|dos mass|seebeck mass',regex=True)
mass_property_rows=curves.loc[mass_mask]
mass_property_rows.to_csv(OUT/'raw_mass_property_candidates.csv',index=False)
print('Sample IDs with S and some carrier/Hall information:',int(samples.broad_overlap.sum()))
print('Raw curve records naming an effective mass:',len(mass_property_rows))
print('An effective-mass label alone does not establish DOS definition or independence from S.')
display(property_audit[property_audit.prop_y.astype(str).str.contains('Seebeck|Carrier concentration|Hall',case=False)]
        .sort_values('curves',ascending=False).head(20))

## Convert only explicitly supported units and axes
The main audit uses `Temperature` or `T` axes in K, and explicitly labelled Celsius axes/units.
Inverse-temperature plots, carrier-concentration-axis Pisarenko plots, and ambiguous units are **listed
for manual recovery**, not silently interpreted. This makes the automatic counts conservative.

Carrier-density curves are converted to cm⁻³. Hall coefficients in m³/C or cm³/C are converted to
the **Hall density** (n_H=1/(e|R_H|)), not asserted to be true (n). No Hall factor is assumed to
turn (n_H) into true (n). Carrier density recorded by the database may also be Hall-derived;
the original method still needs checking. Keep these distinctions in the final baseline.

In [ ]:
# 6. Explicit parsing and unit conversion. Every rejected curve gets a reason.
E_CHARGE=1.602176634e-19
def unit(v):
    return str(v).lower().replace(' ','').replace('**','^').replace('(','').replace(')','').replace('μ','u').replace('µ','u')
def array(v):
    try:
        a=json.loads(v) if isinstance(v,str) else v
    except (ValueError,TypeError):
        a=ast.literal_eval(v)
    return np.asarray(a,dtype=float).reshape(-1)

def extract_point_records(curve_frame):
    records=[]; rejected=[]
    meta=samples.set_index('sample_id')
    eligible_ids=set(samples.loc[samples.eligible_metadata,'sample_id'])
    relevant=curve_frame[curve_frame.py.isin({'seebeck coefficient','hall coefficient'}|n_names)]
    for r in relevant.itertuples(index=False):
        reason=None
        if r.sample_id not in eligible_ids: reason='sample_metadata_ineligible'
        elif r.DOI != meta.at[r.sample_id,'DOI']: reason='curve_sample_DOI_mismatch'
        ux,uy=unit(r.unit_x),unit(r.unit_y)
        if reason is None and r.px not in {'temperature','t','temperature (oc)','temperature (°c)'}:
            reason='non_temperature_axis_manual_recovery'
        if reason is None and ux not in {'k','c','°c','degc'}:
            reason='unsupported_temperature_unit'
        if r.py=='seebeck coefficient':
            kind='S'; scale={'v*k^-1':1e6,'v/k':1e6,'uv/k':1.,'uv*k^-1':1.,'mv/k':1e3}.get(uy)
        elif r.py=='hall coefficient':
            kind='Hall'; scale={'m^3*c^-1':1.,'m^3/c':1.,'cm^3/c':1e-6,'cm^3*c^-1':1e-6}.get(uy)
        else:
            kind='n'; scale={'m^-3':1e-6,'1/m^3':1e-6,'cm^-3':1.,'1/cm^3':1.}.get(uy)
        if reason is None and scale is None: reason='unsupported_property_unit'
        if reason:
            rejected.append({'curve_id':r.curve_id,'sample_id':r.sample_id,'DOI':r.DOI,'reason':reason,
                             'prop_x':r.prop_x,'prop_y':r.prop_y,'unit_x':r.unit_x,'unit_y':r.unit_y})
            continue
        try:
            x,y=array(r.x),array(r.y)
            if len(x)!=len(y) or not len(x): raise ValueError('empty or mismatched arrays')
        except Exception:
            rejected.append({'curve_id':r.curve_id,'sample_id':r.sample_id,'DOI':r.DOI,'reason':'invalid_arrays'})
            continue
        if ux in {'c','°c','degc'}: x=x+273.15
        values=y*scale
        if kind=='Hall':
            with np.errstate(divide='ignore',invalid='ignore'): values=1/(E_CHARGE*np.abs(values))/1e6
        elif kind=='n': values=np.abs(values)
        finite=np.isfinite(x)&np.isfinite(values)&(x>0)
        for j in np.flatnonzero(finite):
            records.append({'sample_id':r.sample_id,'DOI':r.DOI,'curve_id':r.curve_id,'point_index':int(j),
                            'kind':kind,'T_K':float(x[j]),'value':float(values[j]),
                            'source_sign':int(np.sign(y[j]))})
    return pd.DataFrame(records,columns=['sample_id','DOI','curve_id','point_index','kind','T_K','value','source_sign']),pd.DataFrame(rejected,columns=['curve_id','sample_id','DOI','reason','prop_x','prop_y','unit_x','unit_y'])

points,rejected_curves=extract_point_records(curves)
rejected_curves.to_csv(OUT/'excluded_curves_for_manual_recovery.csv',index=False)
points=points.drop_duplicates(['sample_id','DOI','kind','T_K','value','source_sign'])
points['in_study_domain']=points.T_K.between(TEMPERATURE_MIN_K,TEMPERATURE_MAX_K)
points['numeric_screen_ok']=np.where(points.kind.eq('S'),points.value.abs().le(MAX_ABS_SEEBECK_UVK),
                                    points.value.between(MIN_CARRIER_CM3,MAX_CARRIER_CM3))
points.loc[~(points.in_study_domain & points.numeric_screen_ok)].to_csv(OUT/'excluded_points.csv',index=False)
usable_points=points[points.in_study_domain & points.numeric_screen_ok].copy()
display(points.groupby('kind').agg(points=('value','size'),samples=('sample_id','nunique'),papers=('DOI','nunique')))
display(rejected_curves.reason.value_counts().to_frame('excluded_curve_records'))

## Pair measurements from the same sample and paper
For each digitized S point, find a measured carrier point within the specified temperature tolerance.
Prefer an eligible density point; use Hall-derived density only when no density point meets the tolerance.
There is **no extrapolation, no temperature-independent extension of room-temperature n, and no
composition-only matching**. Repeated identical points are removed.

Conflicting carrier curves near the best temperature and Seebeck/Hall sign disagreements are flagged.
Flagged pairs remain visible but are excluded from the restricted candidate set. This is conservative:
multiband behaviour can cause sign differences, so exclusion is not a claim that the source is wrong.
The final number of temperature points remains correlated within samples and papers.

In [ ]:
# 7. Pairing function, reused for sensitivity checks.
S_points=usable_points[usable_points.kind.eq('S')]
N_points=usable_points[usable_points.kind.ne('S')]
n_groups={key:g for key,g in N_points.groupby(['sample_id','DOI'],sort=False)}
pair_columns=['sample_id','DOI','S_curve_id','S_point_index','n_curve_id','T_S_K','T_n_K',
              'temperature_gap_K','S_uV_per_K','carrier_density_cm3','carrier_source',
              'carrier_disagreement','Hall_Seebeck_sign_disagreement','candidate_pair_ok']

def match_points(tolerance):
    rows=[]
    for key, sg in S_points.groupby(['sample_id','DOI'],sort=False):
        ng=n_groups.get(key)
        if ng is None: continue
        for sr in sg.itertuples(index=False):
            distances=np.abs(ng.T_K.to_numpy()-sr.T_K)
            candidates=ng.loc[distances<=tolerance].copy()
            if candidates.empty: continue
            if candidates.kind.eq('n').any(): candidates=candidates[candidates.kind.eq('n')]
            candidates['gap']=np.abs(candidates.T_K-sr.T_K)
            chosen=candidates.sort_values(['gap','curve_id','point_index']).iloc[0]
            nearby=candidates[candidates.gap<=chosen.gap+1.0]
            disagreement=(nearby.value.max()/nearby.value.min()-1)>MAX_RELATIVE_CARRIER_DISAGREEMENT
            sign_disagreement=(chosen.kind=='Hall' and sr.value!=0 and
                               int(np.sign(sr.value))!=int(chosen.source_sign))
            rows.append([key[0],key[1],sr.curve_id,sr.point_index,int(chosen.curve_id),sr.T_K,chosen.T_K,
                         chosen.gap,sr.value,chosen.value,chosen.kind,bool(disagreement),
                         bool(sign_disagreement),not(disagreement or sign_disagreement)])
    result=pd.DataFrame(rows,columns=pair_columns)
    for flag in ['carrier_disagreement','Hall_Seebeck_sign_disagreement','candidate_pair_ok']:
        result[flag]=result[flag].astype(bool)
    return result

pairs=match_points(TEMPERATURE_TOLERANCE_K)
pairs=pairs.merge(samples[['sample_id','DOI','composition','family','process','data_type','form','purity',
                           'explicit_experiment','pool_included','selected_family']],
                  on=['sample_id','DOI'],how='left',validate='many_to_one')
pairs['carrier_type_from_S']=np.select([pairs.S_uV_per_K<0,pairs.S_uV_per_K>0],['n','p'],default='zero_S')
pairs['near_300K']=pairs.T_S_K.sub(ROOM_TEMPERATURE_K).abs().le(ROOM_WINDOW_K)
matched=pairs[pairs.candidate_pair_ok].copy()
print('All matched points:',len(pairs),'Restricted candidate points:',len(matched))
print('Restricted candidate sample IDs:',matched.sample_id.nunique(),'Source papers:',matched.DOI.nunique())
print('These are S/carrier pairs, not yet independent-mass-complete SPB residuals.')
display(pairs.head(10))

In [ ]:
# 8. Compare all available families, the selected pool, and your single family.
summary_rows=[]
for family,sm in samples.groupby('family',sort=True):
    ids=set(sm.sample_id); pp=matched[matched.sample_id.isin(ids)]
    summary_rows.append({'family':family,'metadata_samples':len(sm),
        'broad_S_carrier_overlap_samples':int(sm.broad_overlap.sum()),
        'matched_samples':pp.sample_id.nunique(),'matched_papers':pp.DOI.nunique(),
        'matched_composition_strings':pp.composition.nunique(),'matched_temperature_points':len(pp),
        'near_300K_samples':pp.loc[pp.near_300K,'sample_id'].nunique(),
        'n_type_samples':pp.loc[pp.carrier_type_from_S.eq('n'),'sample_id'].nunique(),
        'p_type_samples':pp.loc[pp.carrier_type_from_S.eq('p'),'sample_id'].nunique(),
        'explicit_experiment_samples':pp.loc[pp.explicit_experiment,'sample_id'].nunique(),
        'largest_paper_sample_share':pp[['sample_id','DOI']].drop_duplicates().DOI.value_counts(normalize=True).max()
                                    if len(pp) else np.nan})
family_summary=pd.DataFrame(summary_rows).sort_values(['matched_samples','matched_papers'],ascending=False)
family_summary.to_csv(OUT/'family_comparison.csv',index=False)
display(family_summary.head(25))
print('Carrier-type counts can overlap if a sample changes S sign with temperature.')

def scope_frame(scope):
    return matched[matched.pool_included] if scope=='Pooled' else matched[matched.selected_family]
scope_rows=[]
for scope in ['Pooled',SELECTED_FAMILY]:
    p=scope_frame(scope)
    scope_rows.append({'scope':scope,'matched_samples':p.sample_id.nunique(),'papers':p.DOI.nunique(),
                      'families':p.family.nunique(),'temperature_points':len(p),
                      'near_300K_samples':p.loc[p.near_300K,'sample_id'].nunique(),
                      'density_curve_samples':p.loc[p.carrier_source.eq('n'),'sample_id'].nunique(),
                      'Hall_conversion_samples':p.loc[p.carrier_source.eq('Hall'),'sample_id'].nunique(),
                      'explicit_experiment_samples':p.loc[p.explicit_experiment,'sample_id'].nunique()})
scope_summary=pd.DataFrame(scope_rows)
display(scope_summary)

In [ ]:
# 9. Check whether the result depends strongly on the matching tolerance.
sensitivity=[]
sample_flags=samples.set_index('sample_id')[['pool_included','selected_family']]
for tol in sorted(set([0.0,2.0,5.0,10.0,float(TEMPERATURE_TOLERANCE_K)])):
    candidate=match_points(tol)
    candidate=candidate[candidate.candidate_pair_ok].join(sample_flags,on='sample_id')
    for scope,flag in [('Pooled','pool_included'),(SELECTED_FAMILY,'selected_family')]:
        q=candidate[candidate[flag].fillna(False)]
        sensitivity.append({'tolerance_K':tol,'scope':scope,'points':len(q),
                            'samples':q.sample_id.nunique(),'papers':q.DOI.nunique()})
tolerance_summary=pd.DataFrame(sensitivity)
tolerance_summary.to_csv(OUT/'temperature_tolerance_sensitivity.csv',index=False)
display(tolerance_summary)

## Effective mass: the remaining input must be explicit
The raw mass-property search does not validate the meaning or origin of those masses. This notebook
**does not use conductivity mass as DOS mass**, nor derive a mass from the Seebeck values being tested.

Optional `independent_dos_mass.csv` columns:
`sample_id, mstar_dos_me, source_doi, method, independent_of_target, T_min_K, T_max_K`.
Here `mstar_dos_me` is (m^*_{DOS}/m_e); `independent_of_target` must be `true`, and the valid temperature
range must cover the matched point. Use one documented assignment per sample in this template.
This declaration requires review; the code cannot verify the paper's measurement/calculation method.
For multiple temperature-dependent masses, use a separately validated mass model instead.

A family mass calibrated only on training studies is a possible **different study design**. It must be
evaluated inside grouped validation; do not call it independently measured mass or use test S to fit it.

In [ ]:
# 10. Optional mass provenance audit; otherwise export a blank assignment template.
mass_cols=['sample_id','mstar_dos_me','source_doi','method','independent_of_target','T_min_K','T_max_K']
mass_template=matched[['sample_id','DOI','composition','family']].drop_duplicates('sample_id').copy()
for col in mass_cols[1:]: mass_template[col]=''
mass_template.to_csv(OUT/'independent_dos_mass_template.csv',index=False)
masses=pd.DataFrame(columns=mass_cols)
if UPLOAD_INDEPENDENT_MASS_TABLE:
    from google.colab import files
    uploaded_mass=files.upload()
    if len(uploaded_mass)!=1: raise ValueError('Upload one independent DOS-mass CSV.')
    name,data=next(iter(uploaded_mass.items()))
    mp=RAW/Path(name).name; mp.write_bytes(data)
    run_info['mass_input']={'name':mp.name,'sha256':sha256(mp)}
    masses=pd.read_csv(mp,dtype={'sample_id':str})
    if not set(mass_cols).issubset(masses.columns): raise ValueError(f'Required columns: {mass_cols}')
    masses=masses[mass_cols].copy()
    masses['sample_id']=masses.sample_id.str.replace(r'\.0$','',regex=True)
    if masses.sample_id.duplicated().any(): raise ValueError('Multiple mass assignments per sample require explicit resolution.')
    for col in ['mstar_dos_me','T_min_K','T_max_K']: masses[col]=pd.to_numeric(masses[col],errors='coerce')
    masses['independent_declared']=masses.independent_of_target.map(norm).isin(['true','yes','1'])
    masses['mass_assignment_complete']=(masses.mstar_dos_me.gt(0)&masses.independent_declared &
        masses.source_doi.map(normalize_doi).str.match(r'^10\.\d{4,9}/\S+$',na=False) &
        masses.method.fillna('').str.strip().ne('') & masses.T_max_K.ge(masses.T_min_K))
    masses.to_csv(OUT/'mass_assignment_audit.csv',index=False)
    matched=matched.merge(masses,on='sample_id',how='left',validate='many_to_one')
    matched['mass_input_complete']=(matched.mass_assignment_complete.fillna(False) &
        matched.T_S_K.ge(matched.T_min_K)&matched.T_S_K.le(matched.T_max_K))
else:
    matched['mass_input_complete']=False

for scope in ['Pooled',SELECTED_FAMILY]:
    q=scope_frame(scope)
    complete=q[q.mass_input_complete]
    ix=scope_summary.scope.eq(scope)
    scope_summary.loc[ix,'mass_input_complete_samples']=complete.sample_id.nunique()
    scope_summary.loc[ix,'mass_input_complete_papers']=complete.DOI.nunique()
scope_summary.to_csv(OUT/'scope_summary.csv',index=False)
display(scope_summary)
print('Mass-input-complete means a documented assignment is present, not that its validity has been proved.')
print('Scattering assumptions, carrier/Hall interpretation, and sample/source verification remain required.')

In [ ]:
# 11. Study concentration, process coverage, and possible specimen duplication.
paper_counts=matched.groupby(['family','DOI']).agg(samples=('sample_id','nunique'),points=('T_S_K','size')).reset_index()
paper_counts.to_csv(OUT/'family_paper_counts.csv',index=False)
process_counts=matched.groupby(['family','process']).agg(samples=('sample_id','nunique'),papers=('DOI','nunique')).reset_index()
process_counts.to_csv(OUT/'family_process_counts.csv',index=False)
print('Selected-family source papers:')
display(paper_counts[paper_counts.family.str.casefold().eq(SELECTED_FAMILY.casefold())]
        .sort_values('samples',ascending=False))
print('Selected-family processing coverage (observational labels, not causal effects):')
display(process_counts[process_counts.family.str.casefold().eq(SELECTED_FAMILY.casefold())])

candidate_samples=matched[['sample_id','DOI','composition','family','process','data_type','form','purity']].drop_duplicates('sample_id')
dup_keys=candidate_samples.groupby(['DOI','composition'],dropna=False).sample_id.nunique()
dup_keys=dup_keys[dup_keys>1].reset_index(name='sample_ids_same_paper_composition')
dup_keys.to_csv(OUT/'same_paper_composition_review.csv',index=False)
print('Same paper/composition with multiple sample IDs:',len(dup_keys),
      '(may be legitimate processing variants; review, do not automatically delete).')
candidate_samples.to_csv(OUT/'candidate_samples.csv',index=False)

## How to decide whether there is enough data
There is no defensible universal rule such as “15 points is enough for SR.” Read the **independent
sample and paper counts**, family balance, carrier-type coverage, and mass-input coverage together.

- **Pooled model:** define its domain using the actually represented families. A model over several
  semiconductor families is not automatically universal across all thermoelectrics.
- **Single-family model:** train on that family's training papers and test on its held-out papers.
- **Fair comparison:** compare the pooled and family-specific models on the **same held-out papers
  in the selected family**, with the same allowed inputs. Remove those papers from both training sets.
- **Stronger transfer claim:** leave a complete family out of pooled training. This is a separate
  experiment from fitting a family-specific equation.
- **Adequacy:** after curating baseline inputs, use grouped learning curves, independent-study tests,
  equation stability across seeds/resamples, and simple competing baselines. Counts alone cannot
  guarantee that a recoverable physical correction exists.

The next table is a feasibility inventory, not a statistical pass/fail threshold.

In [ ]:
# 12. Structural holdout support: how much data remains if one paper is held out?
holdout_rows=[]
for scope in ['Pooled',SELECTED_FAMILY]:
    q=scope_frame(scope)
    unique=q[['sample_id','DOI','family']].drop_duplicates()
    for doi,g in unique.groupby('DOI'):
        train=unique[unique.DOI.ne(doi)]
        holdout_rows.append({'scope':scope,'held_out_DOI':doi,'test_samples':g.sample_id.nunique(),
                            'train_samples':train.sample_id.nunique(),'train_papers':train.DOI.nunique(),
                            'train_families':train.family.nunique()})
holdout=pd.DataFrame(holdout_rows,columns=['scope','held_out_DOI','test_samples','train_samples','train_papers','train_families'])
holdout.to_csv(OUT/'leave_one_paper_out_support.csv',index=False)
if len(holdout):
    display(holdout.groupby('scope').agg(holdouts=('held_out_DOI','size'),
            smallest_training_set=('train_samples','min'),largest_test_paper=('test_samples','max')))
else: print('No paired records survived. Inspect exclusions, units, axes, and the chosen study domain.')

In [ ]:
# 13. Export a reproducible audit package. No raw input files are included in the ZIP.
matched.to_csv(OUT/'candidate_paired_points.csv',index=False)
pairs.loc[~pairs.candidate_pair_ok].to_csv(OUT/'flagged_pairs.csv',index=False)
samples[['sample_id','DOI','composition','family','process','data_type','eligible_metadata',
         'pool_included','has_S_any_axis','has_carrier_or_Hall_any_axis','broad_overlap']].to_csv(
             OUT/'sample_availability.csv',index=False)
run_info['configuration']={k:globals()[k] for k in [
    'DATA_SOURCE','SELECTED_FAMILY','POOLED_FAMILIES','INCLUDE_UNTAGGED_IN_POOL',
    'REQUIRE_EXPLICIT_EXPERIMENT_TAG','TEMPERATURE_TOLERANCE_K','TEMPERATURE_MIN_K','TEMPERATURE_MAX_K',
    'ROOM_TEMPERATURE_K','ROOM_WINDOW_K','MAX_RELATIVE_CARRIER_DISAGREEMENT',
    'MIN_CARRIER_CM3','MAX_CARRIER_CM3','MAX_ABS_SEEBECK_UVK','UPLOAD_INDEPENDENT_MASS_TABLE']}
run_info['raw_counts']={'samples':len(samples),'curves':len(curves)}
run_info['software']={'pandas':pd.__version__,'numpy':np.__version__}
run_info['scope_results']=json.loads(scope_summary.to_json(orient='records'))
(OUT/'run_manifest.json').write_text(json.dumps(run_info,indent=2),encoding='utf-8')

report=['# Pooled and single-family SPB data availability audit','',
        'This report counts candidate records. It does not certify scientific adequacy or source accuracy.','',
        markdown_table(scope_summary),'',
        '## What is established',
        f'- Source snapshot: {run_info.get("snapshot","not supplied; see input file hashes")}.',
        f'- Nearest-point temperature tolerance: {TEMPERATURE_TOLERANCE_K} K; no extrapolation.',
        f'- Explicit study domain: {TEMPERATURE_MIN_K}–{TEMPERATURE_MAX_K} K.',
        '- Sample IDs and source papers are counted separately from correlated curve points.',
        '- Density curves and Hall-derived density are labelled separately.',
        '- Automatic family assignments use explicit database tags, not substring matches.','',
        '## What remains unresolved',
        '- Source-paper confirmation of specimen identity, theory/experiment status, units and phase.',
        '- Independent DOS mass or a separately justified training-only calibration design.',
        '- Hall-factor/scattering conventions and applicability of a single-carrier baseline.',
        '- Duplicate specimens across papers and multiple entries for the same experiment.',
        '- Statistical sufficiency: evaluate grouped learning curves and equation stability.','',
        '## Prioritized next step',
        'Use family_paper_counts.csv to retrieve source papers for the selected family. Verify carrier measurements ',
        'and independent DOS-mass provenance before constructing residual targets. Preserve the pooled comparison.']
(OUT/'audit_report.md').write_text('\n'.join(report),encoding='utf-8')
archive=ROOT/'spb_data_audit_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.glob('*')): z.write(path,path.name)
print('Saved',archive)
display(Markdown('\n'.join(report)))

In [ ]:
# 14. Download results in Colab.
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Outside Colab: results are at',archive.resolve())

## Scientific references and scope
- Ricci et al. (2017), *An ab initio electronic transport database for inorganic materials*.
  DOI: [10.1038/sdata.2017.85](https://doi.org/10.1038/sdata.2017.85).
  Prescribed carrier conditions and conductivity masses do not supply a sample's experimental n and DOS mass.
- de Boor (2021), *On the applicability of the single parabolic band model to advanced thermoelectric
  materials with complex band structures*. DOI: [10.1016/j.jmat.2020.10.013](https://doi.org/10.1016/j.jmat.2020.10.013).
  Hall interpretation and fitted parameters can be misleading in multiband systems.
- Snyder et al. (2022), *Effective Mass from Seebeck Coefficient*.
  DOI: [10.1002/adfm.202112772](https://doi.org/10.1002/adfm.202112772).
  A mass derived from the same target S is not independent evidence for testing that S.
- Witman and Schindler (2025), *MatFold: systematic insights into materials discovery models’ performance
  through standardized cross-validation protocols*. DOI: [10.1039/D4DD00250D](https://doi.org/10.1039/D4DD00250D).
  Supports validation matched to the claimed form of generalization; paper-level grouping here is our
  adaptation to literature-derived repeated measurements.
- Starrydata, *Starrydata: from published plots to shared materials data* (2025).
  DOI: [10.1080/27660400.2025.2506976](https://doi.org/10.1080/27660400.2025.2506976).

Temperature tolerances, numerical screens and the study domain above are transparent **audit choices**,
not thresholds asserted by these papers. The notebook deliberately gives no automatic “thesis approved” verdict.